This is a simple experiment using Retrieval-Augmented Generation (RAG), a commonly used technique in combination with LLMs to overcome weaknesses inherent to them, such as their limitations on knowledge outside of their training set, and preventing them from hallucination (especially when users are also unfamiliar with the topic).

Given the current job market, I thought it might be more relevant to showcase a use case of using RAG on a list of job postings.

In [1]:
import requests
import json

# Scraping jobs from MyCareersFuture API
def scrape_jobs(pages=0, job_title="data scientist"):
    jobs = []
    for page in range(pages):
        url = "https://api.mycareersfuture.gov.sg/v2/search"
    
        headers = {
            "Content-Type": "application/json",
            "User-Agent": "Mozilla/5.0"
        }
    
        payload = {
            "sessionId": "",
            "search": f"{job_title}",
            "postingCompany": [],
            "skillUuids": [
            ],
            "page": page
        }
    
        response = requests.post(url, headers=headers, json=payload)
        
        if response.status_code == 200:
            data = response.json()
            for job in data.get("results", []):
                title = job.get("title")
                uuid = job.get("uuid")
                url = f"https://api.mycareersfuture.gov.sg/v2/jobs/{uuid}?updateApplicationCount=true"
                desc = requests.get(url).json().get("description")
                company = job.get("postedCompany", {}).get("name")
                salary = job.get("salary", {}).get("minimum")
                #print(f"{title} | {company} | Salary: {salary}")
                jobs.append({"title":title,"company":company,"salary":salary,"desc":desc})
        else:
            print("Error:", response.status_code)
            print(response.text)
    return jobs

jobs = scrape_jobs(pages=5)

In [2]:
len(jobs)

100

In [3]:
from langchain.vectorstores import FAISS
from langchain.embeddings import SentenceTransformerEmbeddings
from langchain.llms import Ollama
from langchain.chains import RetrievalQA
from langchain.schema import Document

In [4]:
# Setting up langchain schema
documents = [
    Document(
        page_content=f"{job['title']} at {job['company']} (Salary: {job['salary']})\n\n{job['desc']}",
        metadata={"title": job["title"], "company": job["company"]}
    )
    for job in jobs
]

In [5]:
# Creating vector store
embedding_model = SentenceTransformerEmbeddings(model_name="all-MiniLM-L6-v2")
vectorstore = FAISS.from_documents(documents, embedding_model)

C:\Users\wtan0\AppData\Local\Temp\ipykernel_25124\3981708429.py:2: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-huggingface package and should be used instead. To use it run `pip install -U :class:`~langchain-huggingface` and import as `from :class:`~langchain_huggingface import HuggingFaceEmbeddings``.
  embedding_model = SentenceTransformerEmbeddings(model_name="all-MiniLM-L6-v2")
C:\Users\wtan0\anaconda3\envs\pb\lib\site-packages\torch\nn\modules\module.py:1762: FutureWarning: `encoder_attention_mask` is deprecated and will be removed in version 4.55.0 for `BertSdpaSelfAttention.forward`.
  return forward_call(*args, **kwargs)


In [6]:
# Set up LLM with Ollama
llm = Ollama(model="gemma3")
!ollama pull gemma3

# Build RetrievalQA chain
qa = RetrievalQA.from_chain_type(
    llm=llm,
    retriever = vectorstore.as_retriever(search_type="mmr", search_kwargs={"k": 5}),
    return_source_documents=True
)

C:\Users\wtan0\AppData\Local\Temp\ipykernel_25124\3761123726.py:2: LangChainDeprecationWarning: The class `Ollama` was deprecated in LangChain 0.3.1 and will be removed in 1.0.0. An updated version of the class exists in the :class:`~langchain-ollama package and should be used instead. To use it run `pip install -U :class:`~langchain-ollama` and import as `from :class:`~langchain_ollama import OllamaLLM``.
  llm = Ollama(model="gemma3")  # Or "llama3", "gemma", etc.
pulling manifest â ‹ pulling manifest â ™ pulling manifest â ¹ pulling manifest â ¸ pulling manifest â ¼ pulling manifest â ´ pulling manifest â ¦ pulling manifest â § pulling manifest â ‡ pulling manifest â � pulling manifest â ‹ pulling manifest â ™ pulling manifest â ¹ pulling manifest â ¸ pulling manifest â ¼ pulling manifest 
pulling aeda25e63ebd: 100% â–•â–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–� 3.3 GB                         
pulling e0a42594d802: 100% â–•â–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ

In [7]:
# --- Step 6: Ask questions ---
query = "What are the most common skills required for a data scientist?"
response = qa(query)

print("Answer:\n", response["result"])
print("\nRelevant Documents:")
for doc in response["source_documents"]:
    print("-", doc.metadata["title"], "at", doc.metadata["company"])

C:\Users\wtan0\AppData\Local\Temp\ipykernel_25124\1427340641.py:3: LangChainDeprecationWarning: The method `Chain.__call__` was deprecated in langchain 0.1.0 and will be removed in 1.0. Use :meth:`~invoke` instead.
  response = qa(query)
C:\Users\wtan0\anaconda3\envs\pb\lib\site-packages\torch\nn\modules\module.py:1762: FutureWarning: `encoder_attention_mask` is deprecated and will be removed in version 4.55.0 for `BertSdpaSelfAttention.forward`.
  return forward_call(*args, **kwargs)


Answer:
 Okay, let's analyze the job descriptions to determine the most common skills required for a data scientist, based on the provided information. Here's a breakdown, categorized by frequency and importance:

**1. Core Statistical & Analytical Skills (Very Frequent - Found in All Roles):**

*   **Statistical Knowledge:**  A strong understanding of statistical techniques is *essential*. Specifically, the job descriptions repeatedly mention:
    *   Logistic Regression
    *   Classification Models
    *   Cluster Analysis
    *   Neural Networks
    *   Random Forests
    *   Ensembles
*   **Data Analysis & Interpretation:** The overarching theme of "extracting insights from data" highlights the need for someone who can analyze and interpret data to identify trends and patterns.
*   **Data Wrangling/Cleaning:** Filter and cleanse unstructured (or ambiguous) data into usable data sets that can be analysed.

**2. Programming & Database Skills (Very Frequent - Found in All Roles):**

